# Cell 01 - Load index, model và cấu hình cho Corrective RAG

## Mục tiêu cell này
Load lại toàn bộ thành phần cần thiết để xây dựng Corrective RAG.

## Vì sao cần làm bước này?
Ở notebook 08, ta đã chọn pipeline retrieval tốt nhất:

- `company_handbook`: dùng Dense Retrieval
- `legal`: dùng Weighted Hybrid Retrieval + Reranker

Tuy nhiên, RAG mạnh không chỉ là tìm context tốt.  
Một hệ thống RAG doanh nghiệp còn phải biết khi nào **không nên trả lời**.

Corrective RAG sẽ kiểm tra:
- câu hỏi có nằm trong phạm vi doanh nghiệp/pháp luật không
- context có đủ mạnh không
- có đủ nguồn nội bộ và nguồn pháp luật không
- điểm retrieval/rerank có đủ tin cậy không
- có nên trả lời, hỏi lại, hay từ chối

## Giải thích code
Code sẽ:
1. Load BM25 index
2. Load FAISS dense index
3. Load metadata chunks
4. Load embedding model `intfloat/multilingual-e5-base`
5. Load reranker model `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`
6. Load cấu hình best reranker từ notebook 08
7. Kiểm tra số lượng chunks và vectors có khớp nhau không

## Output mong đợi
Bạn cần thấy:
- chunks khoảng 91,448 dòng
- FAISS vectors = 91,448
- device = cuda nếu GPU hoạt động
- best method = Reranker

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import pickle
import re
import math
import faiss
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

BM25_DIR = PROJECT / "indexes" / "bm25"
FAISS_DIR = PROJECT / "indexes" / "faiss"
METRIC_DIR = PROJECT / "artifacts" / "metrics"
PRED_DIR = PROJECT / "artifacts" / "predictions"
PROMPT_DIR = PROJECT / "artifacts" / "prompts"

METRIC_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

with open(BM25_DIR / "bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

chunks_df = pd.read_csv(FAISS_DIR / "dense_metadata.csv", low_memory=False)
faiss_index = faiss.read_index(str(FAISS_DIR / "faiss_index.index"))

id_cols = ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "source_path", "language"]

for col in id_cols:
    chunks_df[col] = chunks_df[col].fillna("").astype(str)

chunks_df["retrieval_text"] = chunks_df["retrieval_text"].fillna("").astype(str)
chunks_df["chunk_text"] = chunks_df["chunk_text"].fillna("").astype(str)

device = "cuda" if torch.cuda.is_available() else "cpu"

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
reranker_model = CrossEncoder(RERANKER_MODEL_NAME, device=device)

best_config_path = METRIC_DIR / "best_reranker_config.json"

with open(best_config_path, "r", encoding="utf-8") as f:
    best_reranker_config = json.load(f)

print("Project:", PROJECT)
print("Chunks metadata:", chunks_df.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("Device:", device)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Reranker model:", RERANKER_MODEL_NAME)
print("Best method:", best_reranker_config["best_method"])
print("Selected by:", best_reranker_config["selected_by"])
print("Best value:", best_reranker_config["best_value"])

assert len(chunks_df) == faiss_index.ntotal, "Metadata và FAISS index không khớp số lượng."

W0511 02:00:37.050000 26100 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Chunks metadata: (91448, 11)
FAISS vectors: 91448
Device: cuda
Embedding model: intfloat/multilingual-e5-base
Reranker model: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Best method: Reranker
Selected by: recall@5
Best value: 0.784195


# Cell 02 - Tạo hàm retrieval tốt nhất cho Corrective RAG

## Mục tiêu cell này
Tạo lại pipeline retrieval mạnh nhất hiện tại để dùng trong Corrective RAG.

## Vì sao cần làm bước này?
Corrective RAG cần kiểm tra chất lượng context trước khi trả lời.  
Muốn kiểm tra context, trước hết hệ thống phải truy xuất được nguồn từ hai nhóm:

1. `company_handbook`  
   Dùng Dense Retrieval vì handbook tiếng Anh, còn người dùng có thể hỏi tiếng Việt.

2. `legal`  
   Dùng Weighted Hybrid Retrieval để lấy candidate, sau đó dùng Reranker để xếp hạng lại.

## Pipeline trong cell này
Câu hỏi người dùng  
→ lấy source nội bộ bằng Dense Retrieval  
→ lấy source pháp luật bằng Weighted Hybrid + Reranker  
→ gộp hai nhóm nguồn  
→ trả về bảng retrieved sources

## Giải thích code
Code sẽ:
1. Tạo hàm BM25 search
2. Tạo hàm Dense search
3. Tạo hàm Weighted Hybrid RRF
4. Tạo hàm Reranker
5. Tạo hàm `retrieve_cross_policy_sources()`
6. Test thử với câu hỏi quản lý thiết bị làm việc

## Output mong đợi
Bạn cần thấy bảng sources gồm:
- 3 nguồn `company_handbook`
- 3 nguồn `legal`

In [2]:
def bm25_tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-ZÀ-ỹ0-9]+", text)
    return [tok for tok in tokens if len(tok) > 1]


def bm25_search_chunks(query, top_k=50, source_filter=None):
    query_tokens = bm25_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    if source_filter is not None:
        candidate_indices = chunks_df.index[chunks_df["source_type"].isin(source_filter)].to_numpy()
    else:
        candidate_indices = np.arange(len(chunks_df))

    candidate_scores = scores[candidate_indices]
    k = min(top_k, len(candidate_indices))

    top_local = np.argpartition(candidate_scores, -k)[-k:]
    top_local = top_local[np.argsort(candidate_scores[top_local])[::-1]]
    top_indices = candidate_indices[top_local]

    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        rows.append({
            "rank": rank,
            "score": float(scores[idx]),
            "retrieval_method": "bm25",
            "corpus_index": int(idx),
            "corpus_id": row["corpus_id"],
            "chunk_id": row["chunk_id"],
            "parent_id": row["parent_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "language": row["language"],
            "chunk_text": row["chunk_text"]
        })

    return pd.DataFrame(rows)


def dense_search_chunks(query, top_k=50, source_filter=None, search_k=500, max_search_k=None):
    if max_search_k is None:
        max_search_k = len(chunks_df)

    query_embedding = embedding_model.encode(
        ["query: " + str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    current_search_k = min(search_k, len(chunks_df))
    final_rows = []

    while current_search_k <= max_search_k:
        scores, indices = faiss_index.search(query_embedding, current_search_k)

        rows = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            row = chunks_df.iloc[idx]

            if source_filter is not None and row["source_type"] not in source_filter:
                continue

            rows.append({
                "rank": len(rows) + 1,
                "score": float(score),
                "retrieval_method": "dense",
                "corpus_index": int(idx),
                "corpus_id": row["corpus_id"],
                "chunk_id": row["chunk_id"],
                "parent_id": row["parent_id"],
                "source_type": row["source_type"],
                "title": row["title"],
                "language": row["language"],
                "chunk_text": row["chunk_text"]
            })

            if len(rows) >= top_k:
                final_rows = rows
                break

        if len(final_rows) >= top_k:
            break

        current_search_k *= 2

        if current_search_k > max_search_k:
            final_rows = rows
            break

    return pd.DataFrame(final_rows)


def weighted_hybrid_rrf_search_chunks(
    query,
    final_top_k=30,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=None,
    rrf_k=60,
    bm25_weight=0.7,
    dense_weight=1.3
):
    bm25_df = bm25_search_chunks(query, top_k=bm25_top_k, source_filter=source_filter)
    dense_df = dense_search_chunks(
        query,
        top_k=dense_top_k,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    )

    fusion = {}

    for _, row in bm25_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["bm25_rank"] = int(row["rank"])
        fusion[idx]["bm25_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += bm25_weight / (rrf_k + int(row["rank"]))

    for _, row in dense_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["dense_rank"] = int(row["rank"])
        fusion[idx]["dense_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += dense_weight / (rrf_k + int(row["rank"]))

    rows = []

    for item in fusion.values():
        row = chunks_df.iloc[item["corpus_index"]]
        rows.append({
            "rank": None,
            "rrf_score": item["rrf_score"],
            "bm25_rank": item["bm25_rank"],
            "dense_rank": item["dense_rank"],
            "bm25_score": item["bm25_score"],
            "dense_score": item["dense_score"],
            "corpus_index": item["corpus_index"],
            "corpus_id": row["corpus_id"],
            "chunk_id": row["chunk_id"],
            "parent_id": row["parent_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "language": row["language"],
            "chunk_text": row["chunk_text"]
        })

    result_df = pd.DataFrame(rows)
    result_df = result_df.sort_values("rrf_score", ascending=False).head(final_top_k).reset_index(drop=True)
    result_df["rank"] = range(1, len(result_df) + 1)

    return result_df


def rerank_chunks(question, candidate_df, top_k=10, batch_size=16, dedup_parent=True):
    if candidate_df.empty:
        return candidate_df.copy()

    pairs = [(str(question), str(text)) for text in candidate_df["chunk_text"].tolist()]

    rerank_scores = reranker_model.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False
    )

    reranked_df = candidate_df.copy()
    reranked_df["rerank_score"] = rerank_scores
    reranked_df = reranked_df.sort_values("rerank_score", ascending=False).reset_index(drop=True)

    if dedup_parent:
        rows = []
        seen = set()

        for _, row in reranked_df.iterrows():
            parent_id = str(row["parent_id"])

            if parent_id in seen:
                continue

            rows.append(row)
            seen.add(parent_id)

            if len(rows) >= top_k:
                break

        reranked_df = pd.DataFrame(rows).reset_index(drop=True)
    else:
        reranked_df = reranked_df.head(top_k).reset_index(drop=True)

    reranked_df["rank"] = range(1, len(reranked_df) + 1)

    return reranked_df


def retrieve_company_sources(question, top_k=3):
    company_df = dense_search_chunks(
        query=question,
        top_k=top_k,
        source_filter=["company_handbook"],
        search_k=500,
        max_search_k=len(chunks_df)
    ).copy()

    company_df["rrf_score"] = company_df["score"]
    company_df["rerank_score"] = np.nan
    company_df["bm25_rank"] = np.nan
    company_df["dense_rank"] = company_df["rank"]
    company_df["selection_method"] = "dense"
    company_df["selection_score"] = company_df["score"]

    return company_df


def retrieve_legal_sources(question, top_k=3, candidate_top_k=30):
    candidate_df = weighted_hybrid_rrf_search_chunks(
        query=question,
        final_top_k=candidate_top_k,
        bm25_top_k=50,
        dense_top_k=50,
        source_filter=["legal"],
        bm25_weight=0.7,
        dense_weight=1.3
    )

    legal_df = rerank_chunks(
        question=question,
        candidate_df=candidate_df,
        top_k=top_k,
        batch_size=16,
        dedup_parent=True
    ).copy()

    legal_df["selection_method"] = "reranker"
    legal_df["selection_score"] = legal_df["rerank_score"]

    return legal_df


def retrieve_cross_policy_sources(question, company_top_k=3, legal_top_k=3):
    company_df = retrieve_company_sources(question, top_k=company_top_k)
    legal_df = retrieve_legal_sources(question, top_k=legal_top_k)

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    return retrieved_df


test_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

test_sources_df = retrieve_cross_policy_sources(
    question=test_question,
    company_top_k=3,
    legal_top_k=3
)

display(
    test_sources_df[
        [
            "rank",
            "source_type",
            "title",
            "parent_id",
            "chunk_id",
            "selection_method",
            "selection_score",
            "bm25_rank",
            "dense_rank",
            "rerank_score"
        ]
    ]
)

print("Số lượng theo source_type:")
display(test_sources_df["source_type"].value_counts().reset_index())

,rank,source_type,title,parent_id,chunk_id,selection_method,selection_score,bm25_rank,dense_rank,rerank_score
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,dense,0.791058,NaN,1.0,NaN
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_002,dense,0.773632,NaN,2.0,NaN
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_000,dense,0.772039,NaN,3.0,NaN
3,4,legal,legal_cid_211231,211231,legal_211231_000,reranker,0.174754,NaN,5.0,0.174754
4,5,legal,legal_cid_24406,24406,legal_24406_000,reranker,-0.132488,47.0,29.0,-0.132488
5,6,legal,legal_cid_152390,152390,legal_152390_000,reranker,-0.507774,NaN,3.0,-0.507774


Số lượng theo source_type:


,source_type,count
0,company_handbook,3
1,legal,3


# Cell 03 - Tạo Question Router và Corrective Decision Gate

## Mục tiêu cell này
Tạo cơ chế quyết định cho Corrective RAG.

## Vì sao cần làm bước này?
RAG doanh nghiệp không nên luôn luôn trả lời.  
Có những trường hợp hệ thống nên:
- trả lời nếu có đủ context
- trả lời nhưng kèm cảnh báo HR/pháp chế kiểm tra thêm
- hỏi lại nếu câu hỏi quá mơ hồ
- từ chối nếu câu hỏi ngoài phạm vi tài liệu

## Lỗi đã xử lý
Một số câu hỏi có thể không truy xuất được nguồn phù hợp.  
Khi đó DataFrame rỗng có thể không có cột `score`, gây lỗi `KeyError: 'score'`.

Phiên bản này thêm hàm tạo DataFrame rỗng đúng schema để pipeline không bị lỗi.

## Output mong đợi
Bạn cần thấy bảng gồm:
- question
- route
- decision
- reason
- num_sources
- source_distribution

In [4]:
SOURCE_COLUMNS = [
    "rank", "score", "retrieval_method", "corpus_index", "corpus_id",
    "chunk_id", "parent_id", "source_type", "title", "language",
    "chunk_text", "rrf_score", "rerank_score", "bm25_rank",
    "dense_rank", "selection_method", "selection_score"
]

def empty_sources_df():
    return pd.DataFrame(columns=SOURCE_COLUMNS)


def ensure_source_columns(df):
    if df is None or df.empty:
        return empty_sources_df()

    df = df.copy()

    for col in SOURCE_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    return df


def detect_question_route(question):
    q = str(question).lower().strip()
    tokens = re.findall(r"[a-zA-ZÀ-ỹ0-9]+", q)

    vague_patterns = [
        "cái này", "chính sách này", "nó", "vậy thì sao", "áp dụng sao",
        "làm sao đây", "có được không"
    ]

    out_of_scope_terms = [
        "nấu", "phở", "bóng đá", "thời tiết", "du lịch", "game",
        "phim", "nhạc", "món ăn", "nấu ăn"
    ]

    company_terms = [
        "công ty", "company", "handbook", "policy", "chính sách",
        "nhân viên", "work device", "work devices", "thiết bị làm việc",
        "quản lý thiết bị", "laptop", "máy tính", "nội bộ"
    ]

    legal_terms = [
        "luật", "pháp luật", "quy định", "việt nam", "người lao động",
        "hợp đồng lao động", "lương", "thử việc", "bảo hiểm", "nghị định",
        "thông tư", "xử phạt", "trách nhiệm"
    ]

    has_vague = len(tokens) < 4 or any(p in q for p in vague_patterns)
    has_out_scope = any(t in q for t in out_of_scope_terms)
    has_company = any(t in q for t in company_terms)
    has_legal = any(t in q for t in legal_terms)

    if has_out_scope and not has_company and not has_legal:
        return "out_of_scope"

    if has_vague and not has_company and not has_legal:
        return "need_clarification"

    if has_company and has_legal:
        return "cross_policy"

    if has_company:
        return "company_only"

    if has_legal:
        return "legal_only"

    return "unknown"


def retrieve_company_sources_safe(question, top_k=5):
    company_df = dense_search_chunks(
        query=question,
        top_k=top_k,
        source_filter=["company_handbook"],
        search_k=500,
        max_search_k=len(chunks_df)
    )

    company_df = ensure_source_columns(company_df)

    if company_df.empty:
        return company_df

    company_df["rrf_score"] = company_df["score"]
    company_df["rerank_score"] = np.nan
    company_df["bm25_rank"] = np.nan
    company_df["dense_rank"] = company_df["rank"]
    company_df["selection_method"] = "dense"
    company_df["selection_score"] = company_df["score"]

    return company_df


def retrieve_legal_sources_safe(question, top_k=5, candidate_top_k=30):
    legal_df = retrieve_legal_sources(
        question=question,
        top_k=top_k,
        candidate_top_k=candidate_top_k
    )

    legal_df = ensure_source_columns(legal_df)

    if legal_df.empty:
        return legal_df

    legal_df["selection_method"] = "reranker"
    legal_df["selection_score"] = legal_df["rerank_score"]

    return legal_df


def retrieve_sources_by_route(question, route):
    if route == "cross_policy":
        company_df = retrieve_company_sources_safe(question, top_k=3)
        legal_df = retrieve_legal_sources_safe(question, top_k=3, candidate_top_k=30)

        retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
        retrieved_df = ensure_source_columns(retrieved_df)

        if not retrieved_df.empty:
            retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

        return retrieved_df

    if route == "company_only":
        company_df = retrieve_company_sources_safe(question, top_k=5)

        if not company_df.empty:
            company_df["rank"] = range(1, len(company_df) + 1)

        return company_df

    if route == "legal_only":
        legal_df = retrieve_legal_sources_safe(question, top_k=5, candidate_top_k=30)

        if not legal_df.empty:
            legal_df["rank"] = range(1, len(legal_df) + 1)

        return legal_df

    return empty_sources_df()


def assess_context_quality(question, route, retrieved_df):
    retrieved_df = ensure_source_columns(retrieved_df)

    if route == "out_of_scope":
        return {
            "decision": "refuse",
            "reason": "Câu hỏi nằm ngoài phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống."
        }

    if route == "need_clarification":
        return {
            "decision": "ask_clarification",
            "reason": "Câu hỏi quá mơ hồ, cần người dùng nói rõ chính sách/vấn đề cụ thể."
        }

    if route == "unknown":
        return {
            "decision": "ask_clarification",
            "reason": "Chưa xác định được câu hỏi thuộc nhóm handbook, pháp luật hay đối chiếu chính sách."
        }

    if retrieved_df.empty:
        return {
            "decision": "refuse",
            "reason": "Không truy xuất được context phù hợp từ tài liệu."
        }

    company_df = retrieved_df[retrieved_df["source_type"] == "company_handbook"]
    legal_df = retrieved_df[retrieved_df["source_type"] == "legal"]

    company_max = float(company_df["selection_score"].max()) if not company_df.empty else None
    legal_max = float(legal_df["selection_score"].max()) if not legal_df.empty else None

    company_min_score = 0.76
    legal_weak_min_score = -0.75
    legal_strong_min_score = 0.0

    if route == "company_only":
        if company_max is None or company_max < company_min_score:
            return {
                "decision": "refuse",
                "reason": "Nguồn handbook nội bộ có điểm truy xuất thấp, chưa đủ tin cậy để trả lời."
            }

        return {
            "decision": "answer",
            "reason": "Tìm thấy nguồn handbook nội bộ đủ liên quan để trả lời."
        }

    if route == "legal_only":
        if legal_max is None or legal_max < legal_weak_min_score:
            return {
                "decision": "refuse",
                "reason": "Nguồn pháp luật truy xuất yếu, chưa đủ căn cứ để trả lời."
            }

        if legal_max < legal_strong_min_score:
            return {
                "decision": "answer_with_caution",
                "reason": "Có nguồn pháp luật liên quan nhưng điểm reranker chưa cao, cần trả lời thận trọng."
            }

        return {
            "decision": "answer",
            "reason": "Có nguồn pháp luật đủ liên quan để trả lời."
        }

    if route == "cross_policy":
        if company_df.empty:
            return {
                "decision": "ask_clarification",
                "reason": "Thiếu nguồn chính sách nội bộ để đối chiếu."
            }

        if legal_df.empty:
            return {
                "decision": "answer_with_caution",
                "reason": "Có nguồn nội bộ nhưng thiếu nguồn pháp luật Việt Nam, cần HR/pháp chế kiểm tra thêm."
            }

        if company_max is None or company_max < company_min_score:
            return {
                "decision": "ask_clarification",
                "reason": "Nguồn nội bộ truy xuất chưa đủ mạnh, cần hỏi rõ chính sách cụ thể hơn."
            }

        if legal_max is None or legal_max < legal_weak_min_score:
            return {
                "decision": "answer_with_caution",
                "reason": "Có nguồn nội bộ nhưng nguồn pháp luật yếu, chỉ nên trả lời định hướng và khuyến nghị HR/pháp chế kiểm tra."
            }

        return {
            "decision": "answer_with_legal_review_notice",
            "reason": "Có cả nguồn nội bộ và nguồn pháp luật liên quan, nhưng vẫn cần cảnh báo HR/pháp chế kiểm tra trước khi áp dụng chính thức."
        }

    return {
        "decision": "ask_clarification",
        "reason": "Chưa đủ thông tin để quyết định."
    }


def corrective_rag_decision(question):
    route = detect_question_route(question)
    retrieved_df = retrieve_sources_by_route(question, route)
    assessment = assess_context_quality(question, route, retrieved_df)

    source_distribution = (
        retrieved_df["source_type"].value_counts().to_dict()
        if not retrieved_df.empty
        else {}
    )

    return {
        "question": question,
        "route": route,
        "decision": assessment["decision"],
        "reason": assessment["reason"],
        "num_sources": len(retrieved_df),
        "source_distribution": source_distribution,
        "retrieved_sources": retrieved_df
    }


test_questions = [
    "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?",
    "What is the company policy for managing work devices?",
    "Lương thử việc được quy định như thế nào?",
    "Cách nấu phở bò ngon tại nhà như thế nào?",
    "Chính sách này áp dụng sao?"
]

decision_outputs = [corrective_rag_decision(q) for q in test_questions]

decision_summary_df = pd.DataFrame([
    {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "reason": item["reason"],
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"]
    }
    for item in decision_outputs
])

display(decision_summary_df)

,question,route,decision,reason,num_sources,source_distribution
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,Có cả nguồn nội bộ và nguồn pháp luật liên qua...,6,"{'company_handbook': 3, 'legal': 3}"
1,What is the company policy for managing work d...,company_only,answer,Tìm thấy nguồn handbook nội bộ đủ liên quan để...,5,{'company_handbook': 5}
2,Lương thử việc được quy định như thế nào?,legal_only,answer,Có nguồn pháp luật đủ liên quan để trả lời.,5,{'legal': 5}
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,Câu hỏi nằm ngoài phạm vi tài liệu doanh nghiệ...,0,{}
4,Chính sách này áp dụng sao?,company_only,refuse,Không truy xuất được context phù hợp từ tài liệu.,0,{}


# Cell 04 - Cải thiện Question Router cho câu hỏi mơ hồ

## Mục tiêu cell này
Sửa lại bộ phân loại câu hỏi để xử lý tốt hơn các câu hỏi mơ hồ.

## Vì sao cần làm bước này?
Ở Cell 03, câu hỏi:

`Chính sách này áp dụng sao?`

bị phân loại thành `company_only` vì có từ `chính sách`.  
Nhưng thực tế câu này không nói rõ là chính sách nào, áp dụng cho ai, trong bối cảnh nào.

Với RAG doanh nghiệp, các câu như vậy không nên bị từ chối ngay, cũng không nên cố truy xuất tài liệu.  
Hệ thống nên hỏi lại người dùng để làm rõ.

## Chiến lược sửa
Nếu câu hỏi có các cụm mơ hồ như:
- `chính sách này`
- `cái này`
- `nó`
- `áp dụng sao`
- `có được không`

và không có đối tượng cụ thể như:
- `thiết bị`
- `lương`
- `thử việc`
- `bảo hiểm`
- `hợp đồng`
- `nhân viên Việt Nam`

thì route sẽ là `need_clarification`.

## Output mong đợi
Câu `Chính sách này áp dụng sao?` phải được chuyển thành:
- route: `need_clarification`
- decision: `ask_clarification`

In [5]:
def detect_question_route(question):
    q = str(question).lower().strip()
    tokens = re.findall(r"[a-zA-ZÀ-ỹ0-9]+", q)

    vague_patterns = [
        "cái này", "chính sách này", "nó", "vậy thì sao", "áp dụng sao",
        "làm sao đây", "có được không"
    ]

    concrete_terms = [
        "thiết bị", "thiết bị làm việc", "quản lý thiết bị", "laptop", "máy tính",
        "lương", "thử việc", "bảo hiểm", "hợp đồng", "người lao động",
        "nhân viên việt nam", "work devices", "managing work devices",
        "remote wipe", "code", "secrets"
    ]

    out_of_scope_terms = [
        "nấu", "phở", "bóng đá", "thời tiết", "du lịch", "game",
        "phim", "nhạc", "món ăn", "nấu ăn"
    ]

    company_terms = [
        "công ty", "company", "handbook", "policy", "chính sách",
        "nhân viên", "work device", "work devices", "thiết bị làm việc",
        "quản lý thiết bị", "laptop", "máy tính", "nội bộ"
    ]

    legal_terms = [
        "luật", "pháp luật", "quy định", "việt nam", "người lao động",
        "hợp đồng lao động", "lương", "thử việc", "bảo hiểm", "nghị định",
        "thông tư", "xử phạt", "trách nhiệm"
    ]

    has_vague = len(tokens) < 4 or any(p in q for p in vague_patterns)
    has_concrete = any(t in q for t in concrete_terms)
    has_out_scope = any(t in q for t in out_of_scope_terms)
    has_company = any(t in q for t in company_terms)
    has_legal = any(t in q for t in legal_terms)

    if has_out_scope and not has_company and not has_legal:
        return "out_of_scope"

    if has_vague and not has_concrete:
        return "need_clarification"

    if has_company and has_legal:
        return "cross_policy"

    if has_company:
        return "company_only"

    if has_legal:
        return "legal_only"

    return "unknown"


test_questions = [
    "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?",
    "What is the company policy for managing work devices?",
    "Lương thử việc được quy định như thế nào?",
    "Cách nấu phở bò ngon tại nhà như thế nào?",
    "Chính sách này áp dụng sao?"
]

decision_outputs = [corrective_rag_decision(q) for q in test_questions]

decision_summary_df = pd.DataFrame([
    {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "reason": item["reason"],
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"]
    }
    for item in decision_outputs
])

display(decision_summary_df)

,question,route,decision,reason,num_sources,source_distribution
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,Có cả nguồn nội bộ và nguồn pháp luật liên qua...,6,"{'company_handbook': 3, 'legal': 3}"
1,What is the company policy for managing work d...,company_only,answer,Tìm thấy nguồn handbook nội bộ đủ liên quan để...,5,{'company_handbook': 5}
2,Lương thử việc được quy định như thế nào?,legal_only,answer,Có nguồn pháp luật đủ liên quan để trả lời.,5,{'legal': 5}
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,Câu hỏi nằm ngoài phạm vi tài liệu doanh nghiệ...,0,{}
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,"Câu hỏi quá mơ hồ, cần người dùng nói rõ chính...",0,{}


# Cell 05 - Tạo Corrective RAG output theo từng decision

## Mục tiêu cell này
Tạo hàm sinh output cuối cùng cho Corrective RAG dựa trên quyết định ở Cell 04.

## Vì sao cần làm bước này?
Ở Cell 04, hệ thống mới quyết định được câu hỏi thuộc loại nào:
- trả lời
- trả lời thận trọng
- trả lời kèm cảnh báo pháp chế
- hỏi lại
- từ chối

Nhưng để dùng trong app thật, ta cần chuyển decision đó thành output rõ ràng cho người dùng.

## Các loại output
Corrective RAG sẽ xử lý như sau:

- `answer`: trả lời dựa trên context
- `answer_with_caution`: trả lời nhưng nói rõ context chưa đủ mạnh
- `answer_with_legal_review_notice`: trả lời kèm cảnh báo HR/pháp chế kiểm tra
- `ask_clarification`: hỏi lại người dùng để làm rõ
- `refuse`: từ chối vì ngoài phạm vi hoặc không có tài liệu phù hợp

## Giải thích code
Code sẽ:
1. Tạo hàm format source/evidence
2. Tạo hàm `build_corrective_rag_output()`
3. Test lại trên 5 câu hỏi mẫu
4. Hiển thị output cuối cùng của từng case

## Output mong đợi
Bạn cần thấy mỗi câu hỏi có:
- route
- decision
- answer
- sources nếu có
- evidence nếu có

In [6]:
def safe_float(value):
    if pd.isna(value):
        return None
    return float(value)


def build_source_records(retrieved_df):
    retrieved_df = ensure_source_columns(retrieved_df)

    sources = []

    for _, row in retrieved_df.iterrows():
        sources.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "parent_id": row["parent_id"],
            "chunk_id": row["chunk_id"],
            "selection_method": row["selection_method"],
            "selection_score": safe_float(row["selection_score"]),
            "bm25_rank": safe_float(row["bm25_rank"]),
            "dense_rank": safe_float(row["dense_rank"]),
            "rerank_score": safe_float(row["rerank_score"])
        })

    return sources


def build_evidence_records(retrieved_df, max_chars=420):
    retrieved_df = ensure_source_columns(retrieved_df)

    evidence = []

    for _, row in retrieved_df.iterrows():
        evidence.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "selection_method": row["selection_method"],
            "evidence": str(row["chunk_text"])[:max_chars]
        })

    return evidence


def build_corrective_rag_output(question):
    decision_item = corrective_rag_decision(question)

    route = decision_item["route"]
    decision = decision_item["decision"]
    reason = decision_item["reason"]
    retrieved_df = decision_item["retrieved_sources"]

    sources = build_source_records(retrieved_df) if not retrieved_df.empty else []
    evidence = build_evidence_records(retrieved_df) if not retrieved_df.empty else []

    if decision == "refuse":
        answer = (
            "Tôi chưa tìm thấy đủ thông tin phù hợp trong phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống. "
            "Vì vậy tôi không nên trả lời để tránh suy đoán sai."
        )

    elif decision == "ask_clarification":
        answer = (
            "Bạn vui lòng nói rõ hơn chính sách hoặc vấn đề cụ thể cần hỏi. "
            "Ví dụ: chính sách thiết bị làm việc, lương thử việc, bảo hiểm, hợp đồng lao động, "
            "hoặc quy định áp dụng cho nhân viên Việt Nam."
        )

    elif decision == "answer":
        if route == "company_only":
            answer = (
                "Tôi tìm thấy nguồn handbook nội bộ liên quan. "
                "Câu trả lời nên được xây dựng dựa trên các evidence bên dưới và chỉ dùng thông tin trong tài liệu nội bộ."
            )
        elif route == "legal_only":
            answer = (
                "Tôi tìm thấy nguồn pháp luật/tài liệu Việt Nam liên quan. "
                "Câu trả lời nên được xây dựng dựa trên các evidence bên dưới và không suy đoán ngoài tài liệu."
            )
        else:
            answer = (
                "Tôi tìm thấy context đủ liên quan để trả lời dựa trên tài liệu được cung cấp."
            )

    elif decision == "answer_with_caution":
        answer = (
            "Tôi tìm thấy một số nguồn liên quan, nhưng độ tin cậy hoặc độ bao phủ chưa đủ mạnh. "
            "Có thể trả lời ở mức định hướng, đồng thời cần kiểm tra thêm với HR/pháp chế trước khi áp dụng."
        )

    elif decision == "answer_with_legal_review_notice":
        answer = (
            "Tôi tìm thấy cả nguồn chính sách nội bộ và nguồn pháp luật/tài liệu Việt Nam liên quan. "
            "Có thể tạo câu trả lời đối chiếu, nhưng vì đây là bối cảnh áp dụng cho nhân viên Việt Nam, "
            "kết quả cần được HR/pháp chế kiểm tra trước khi áp dụng chính thức."
        )

    else:
        answer = (
            "Tôi chưa đủ thông tin để quyết định cách trả lời phù hợp. "
            "Bạn vui lòng cung cấp thêm ngữ cảnh."
        )

    return {
        "question": question,
        "route": route,
        "decision": decision,
        "reason": reason,
        "answer": answer,
        "num_sources": len(retrieved_df),
        "source_distribution": decision_item["source_distribution"],
        "sources": sources,
        "evidence": evidence
    }


corrective_outputs = [build_corrective_rag_output(q) for q in test_questions]

for item in corrective_outputs:
    print("=" * 100)
    print("QUESTION:", item["question"])
    print("ROUTE:", item["route"])
    print("DECISION:", item["decision"])
    print("REASON:", item["reason"])
    print("ANSWER:", item["answer"])
    print("NUM SOURCES:", item["num_sources"])
    print("SOURCE DISTRIBUTION:", item["source_distribution"])

    if item["sources"]:
        print("\nSOURCES:")
        display(pd.DataFrame(item["sources"]))

    if item["evidence"]:
        print("\nEVIDENCE PREVIEW:")
        for ev in item["evidence"][:2]:
            print(f"[Source {ev['rank']}] {ev['source_type']} | {ev['title']}")
            print(ev["evidence"])
            print()

QUESTION: Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
ROUTE: cross_policy
DECISION: answer_with_legal_review_notice
REASON: Có cả nguồn nội bộ và nguồn pháp luật liên quan, nhưng vẫn cần cảnh báo HR/pháp chế kiểm tra trước khi áp dụng chính thức.
ANSWER: Tôi tìm thấy cả nguồn chính sách nội bộ và nguồn pháp luật/tài liệu Việt Nam liên quan. Có thể tạo câu trả lời đối chiếu, nhưng vì đây là bối cảnh áp dụng cho nhân viên Việt Nam, kết quả cần được HR/pháp chế kiểm tra trước khi áp dụng chính thức.
NUM SOURCES: 6
SOURCE DISTRIBUTION: {'company_handbook': 3, 'legal': 3}

SOURCES:


,rank,source_type,title,parent_id,chunk_id,selection_method,selection_score,bm25_rank,dense_rank,rerank_score
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,dense,0.791058,NaN,1.0,NaN
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_002,dense,0.773632,NaN,2.0,NaN
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_000,dense,0.772039,NaN,3.0,NaN
3,4,legal,legal_cid_211231,211231,legal_211231_000,reranker,0.174754,NaN,5.0,0.174754
4,5,legal,legal_cid_24406,24406,legal_24406_000,reranker,-0.132488,47.0,29.0,-0.132488
5,6,legal,legal_cid_152390,152390,legal_152390_000,reranker,-0.507774,NaN,3.0,-0.507774



EVIDENCE PREVIEW:
[Source 1] company_handbook | Managing Work Devices
With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust

[Source 2] company_handbook | Managing Work Devices
## Mobile devices and Windows
Devices running Android, iOS/iPadOS, and Windows are currently unmanaged. It’s fine to install our BC4 and HEY apps on these devices to access work projects and email, but since they’re unmanaged – and therefore ‘untrusted’ – it’s not okay to store 37signals code or secrets on them. If you're coding or accessing secure systems, you should be doing so on a Kandji-managed Mac or an Omarchy

QUESTION: What is the company pol

,rank,source_type,title,parent_id,chunk_id,selection_method,selection_score,bm25_rank,dense_rank,rerank_score
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,dense,0.825828,None,1.0,None
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_000,dense,0.825389,None,2.0,None
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_002,dense,0.819602,None,3.0,None
3,4,company_handbook,Moonlighting,company_0005,company_0005_002,dense,0.788421,None,4.0,None
4,5,company_handbook,Readme,company_0008,company_0008_000,dense,0.783055,None,5.0,None



EVIDENCE PREVIEW:
[Source 1] company_handbook | Managing Work Devices
With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust

[Source 2] company_handbook | Managing Work Devices
# Managing work devices
Everyone receives a new Mac when they join 37signals. We centrally manage and secure these devices with Kandji which reduces our exposure to security incidents. Kandji applies a standard configuration to every device (e.g. enable disk encryption, firewall, password rules), it installs essential apps (e.g. EncryptMe), and it will ensure the apps have the latest security updates applied. Kandji 

QUESTION: Lương thử việc được quy

,rank,source_type,title,parent_id,chunk_id,selection_method,selection_score,bm25_rank,dense_rank,rerank_score
0,1,legal,legal_cid_116782,116782,legal_116782_000,reranker,1.562081,None,1.0,1.562081
1,2,legal,legal_cid_61953,61953,legal_61953_000,reranker,-0.086482,None,2.0,-0.086482
2,3,legal,legal_cid_107047,107047,legal_107047_001,reranker,-1.060560,None,9.0,-1.060560
3,4,legal,legal_cid_175106,175106,legal_175106_000,reranker,-1.423649,None,4.0,-1.423649
4,5,legal,legal_cid_77700,77700,legal_77700_000,reranker,-1.604817,None,29.0,-1.604817



EVIDENCE PREVIEW:
[Source 1] legal | legal_cid_116782
Vi phạm quy định về thử việc
1. Phạt tiền từ 500.000 đồng đến 1.000.000 đồng đối với người sử dụng lao động có một trong các hành vi sau đây:
a) Yêu cầu thử việc đối với người lao động làm việc theo hợp đồng lao động có thời hạn dưới 01 tháng;
...

[Source 2] legal | legal_cid_61953
“Điều 26. Tiền lương trong thời gian thử việc
 Tiền lương của người lao động trong thời gian thử việc do hai bên thoả thuận nhưng ít nhất phải bằng 85% mức lương của công việc đó.”

QUESTION: Cách nấu phở bò ngon tại nhà như thế nào?
ROUTE: out_of_scope
DECISION: refuse
REASON: Câu hỏi nằm ngoài phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống.
ANSWER: Tôi chưa tìm thấy đủ thông tin phù hợp trong phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống. Vì vậy tôi không nên trả lời để tránh suy đoán sai.
NUM SOURCES: 0
SOURCE DISTRIBUTION: {}
QUESTION: Chính sách này áp dụng sao?
ROUTE: need_clarification
DECISION: ask_clarification
REASON: Câu hỏi quá 

# Cell 06 - Tạo prompt cuối cùng cho Corrective RAG

## Mục tiêu cell này
Tạo prompt RAG cuối cùng dựa trên decision của Corrective Gate.

## Vì sao cần làm bước này?
Ở Cell 05, hệ thống đã biết nên trả lời, hỏi lại hay từ chối.  
Tuy nhiên, nếu được phép trả lời, hệ thống cần tạo prompt chuẩn để LLM sinh câu trả lời dựa trên evidence.

Corrective RAG sẽ hoạt động như sau:

- Nếu `decision = answer` → tạo prompt trả lời bình thường
- Nếu `decision = answer_with_caution` → tạo prompt trả lời thận trọng
- Nếu `decision = answer_with_legal_review_notice` → tạo prompt trả lời kèm cảnh báo HR/pháp chế
- Nếu `decision = ask_clarification` → không gọi LLM, hỏi lại người dùng
- Nếu `decision = refuse` → không gọi LLM, từ chối trả lời

## Giải thích code
Code sẽ:
1. Format retrieved sources thành context
2. Tạo prompt theo từng decision
3. Tạo output cuối gồm answer/direct_response hoặc prompt
4. Test lại 5 câu hỏi mẫu
5. Lưu kết quả demo vào JSON

## Output mong đợi
Bạn cần thấy:
- Các câu có thể trả lời sẽ có `should_call_llm = True`
- Câu ngoài phạm vi và câu mơ hồ sẽ có `should_call_llm = False`

In [7]:
def format_corrective_context(retrieved_df, max_chars_per_chunk=1200):
    retrieved_df = ensure_source_columns(retrieved_df)
    blocks = []

    for _, row in retrieved_df.iterrows():
        text = str(row["chunk_text"])[:max_chars_per_chunk]

        selection_score = row["selection_score"]
        selection_score_text = "None" if pd.isna(selection_score) else f"{float(selection_score):.6f}"

        rerank_score = row["rerank_score"]
        rerank_score_text = "None" if pd.isna(rerank_score) else f"{float(rerank_score):.6f}"

        block = f"""
[Source {int(row["rank"])}]
source_type: {row["source_type"]}
title: {row["title"]}
parent_id: {row["parent_id"]}
chunk_id: {row["chunk_id"]}
selection_method: {row["selection_method"]}
selection_score: {selection_score_text}
bm25_rank: {row["bm25_rank"]}
dense_rank: {row["dense_rank"]}
rerank_score: {rerank_score_text}

{text}
""".strip()

        blocks.append(block)

    return "\n\n---\n\n".join(blocks)


def build_corrective_prompt(corrective_output):
    question = corrective_output["question"]
    route = corrective_output["route"]
    decision = corrective_output["decision"]
    reason = corrective_output["reason"]
    retrieved_df = corrective_rag_decision(question)["retrieved_sources"]

    context = format_corrective_context(retrieved_df)

    if decision == "answer":
        instruction = "Trả lời trực tiếp dựa trên CONTEXT. Không suy đoán ngoài tài liệu."
    elif decision == "answer_with_caution":
        instruction = "Trả lời thận trọng dựa trên CONTEXT. Nói rõ rằng nguồn hiện tại chưa đủ mạnh để kết luận chắc chắn."
    elif decision == "answer_with_legal_review_notice":
        instruction = "Trả lời đối chiếu giữa nguồn nội bộ và nguồn pháp luật. Bắt buộc nhắc HR/pháp chế kiểm tra trước khi áp dụng chính thức."
    else:
        instruction = "Không tạo câu trả lời dài. Hãy xử lý theo decision."

    prompt = f"""
Bạn là trợ lý RAG nội bộ doanh nghiệp.

ROUTE:
{route}

DECISION:
{decision}

LÝ DO DECISION:
{reason}

NHIỆM VỤ:
{instruction}

QUY TẮC BẮT BUỘC:
- Chỉ dùng thông tin trong CONTEXT.
- Không bịa thêm điều khoản, luật, chính sách hoặc con số không có trong CONTEXT.
- Nếu nguồn pháp luật chưa đủ chắc, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Câu trả lời phải dễ hiểu, có cấu trúc và có nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
{question}

CONTEXT:
{context}

YÊU CẦU OUTPUT:
1. Câu trả lời
2. Nguồn đã dùng
3. Evidence ngắn gọn
4. Mức độ tin cậy / lưu ý
""".strip()

    return prompt


def build_final_corrective_response(question):
    output = build_corrective_rag_output(question)
    decision = output["decision"]

    if decision in ["refuse", "ask_clarification"]:
        output["should_call_llm"] = False
        output["prompt"] = None
        output["final_response_type"] = "direct_response"
        return output

    output["should_call_llm"] = True
    output["prompt"] = build_corrective_prompt(output)
    output["final_response_type"] = "rag_prompt"
    return output


final_corrective_outputs = [build_final_corrective_response(q) for q in test_questions]

summary_rows = []

for item in final_corrective_outputs:
    summary_rows.append({
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "final_response_type": item["final_response_type"],
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"]
    })

final_corrective_summary_df = pd.DataFrame(summary_rows)

corrective_demo_path = PROMPT_DIR / "corrective_rag_demo_outputs.json"

with open(corrective_demo_path, "w", encoding="utf-8") as f:
    json.dump(final_corrective_outputs, f, ensure_ascii=False, indent=2)

print("Đã lưu:", corrective_demo_path)
display(final_corrective_summary_df)

for item in final_corrective_outputs:
    print("=" * 100)
    print("QUESTION:", item["question"])
    print("DECISION:", item["decision"])
    print("SHOULD CALL LLM:", item["should_call_llm"])
    if item["should_call_llm"]:
        print("PROMPT PREVIEW:")
        print(item["prompt"][:2000])
    else:
        print("DIRECT RESPONSE:")
        print(item["answer"])

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\corrective_rag_demo_outputs.json


,question,route,decision,should_call_llm,final_response_type,num_sources,source_distribution
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,rag_prompt,6,"{'company_handbook': 3, 'legal': 3}"
1,What is the company policy for managing work d...,company_only,answer,True,rag_prompt,5,{'company_handbook': 5}
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,rag_prompt,5,{'legal': 5}
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,direct_response,0,{}
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,direct_response,0,{}


QUESTION: Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
DECISION: answer_with_legal_review_notice
SHOULD CALL LLM: True
PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

ROUTE:
cross_policy

DECISION:
answer_with_legal_review_notice

LÝ DO DECISION:
Có cả nguồn nội bộ và nguồn pháp luật liên quan, nhưng vẫn cần cảnh báo HR/pháp chế kiểm tra trước khi áp dụng chính thức.

NHIỆM VỤ:
Trả lời đối chiếu giữa nguồn nội bộ và nguồn pháp luật. Bắt buộc nhắc HR/pháp chế kiểm tra trước khi áp dụng chính thức.

QUY TẮC BẮT BUỘC:
- Chỉ dùng thông tin trong CONTEXT.
- Không bịa thêm điều khoản, luật, chính sách hoặc con số không có trong CONTEXT.
- Nếu nguồn pháp luật chưa đủ chắc, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Câu trả lời phải dễ hiểu, có cấu trúc và có nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị

Hiện tại Corrective RAG đã biết:

3 câu nên gọi LLM:
- cross_policy
- company_only
- legal_only

2 câu không nên gọi LLM:
- out_of_scope
- need_clarification

Nhưng có một điểm nên sửa trước khi đi tiếp: trong build_corrective_prompt(), code đang gọi lại corrective_rag_decision(question) lần nữa. Như vậy mỗi lần tạo prompt sẽ retrieval lại, gây chậm vì reranker phải chạy lại. Ta nên tạo pipeline final chỉ retrieval một lần.

# Cell 07 - Tối ưu Corrective RAG pipeline để không retrieval lặp lại

## Mục tiêu cell này
Tạo phiên bản Corrective RAG pipeline cuối cùng, chỉ truy xuất context một lần cho mỗi câu hỏi.

## Vì sao cần làm bước này?
Ở Cell 06, hàm tạo prompt đang gọi lại `corrective_rag_decision(question)`.  
Điều này làm hệ thống phải chạy retrieval/reranker lại lần nữa, gây chậm không cần thiết.

Corrective RAG thực tế nên chạy theo luồng:

Question  
→ route câu hỏi  
→ retrieve sources  
→ decision gate  
→ nếu được phép trả lời thì tạo prompt từ chính sources đã retrieve  
→ nếu không được phép thì trả lời trực tiếp

## Giải thích code
Code sẽ:
1. Tạo hàm `build_corrective_prompt_from_sources()`
2. Tạo hàm `corrective_rag_pipeline_final()`
3. Đảm bảo mỗi câu hỏi chỉ retrieve một lần
4. Lưu prompt demo cho case CrossPolicyRAG
5. Lưu output demo final vào JSON

## Output mong đợi
Bạn cần thấy bảng 5 case giống Cell 06, nhưng pipeline gọn hơn và không retrieval lặp lại.

In [8]:
def build_corrective_prompt_from_sources(question, route, decision, reason, retrieved_df):
    context = format_corrective_context(retrieved_df)

    if decision == "answer":
        instruction = "Trả lời trực tiếp dựa trên CONTEXT. Không suy đoán ngoài tài liệu."
    elif decision == "answer_with_caution":
        instruction = "Trả lời thận trọng dựa trên CONTEXT. Nói rõ rằng nguồn hiện tại chưa đủ mạnh để kết luận chắc chắn."
    elif decision == "answer_with_legal_review_notice":
        instruction = "Trả lời đối chiếu giữa nguồn nội bộ và nguồn pháp luật. Bắt buộc nhắc HR/pháp chế kiểm tra trước khi áp dụng chính thức."
    else:
        instruction = "Không tạo câu trả lời dài. Hãy xử lý theo decision."

    prompt = f"""
Bạn là trợ lý RAG nội bộ doanh nghiệp.

ROUTE:
{route}

DECISION:
{decision}

LÝ DO DECISION:
{reason}

NHIỆM VỤ:
{instruction}

QUY TẮC BẮT BUỘC:
- Chỉ dùng thông tin trong CONTEXT.
- Không bịa thêm điều khoản, luật, chính sách hoặc con số không có trong CONTEXT.
- Nếu nguồn pháp luật chưa đủ chắc, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Câu trả lời phải dễ hiểu, có cấu trúc và có nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
{question}

CONTEXT:
{context}

YÊU CẦU OUTPUT:
1. Câu trả lời
2. Nguồn đã dùng
3. Evidence ngắn gọn
4. Mức độ tin cậy / lưu ý
""".strip()

    return prompt


def corrective_rag_pipeline_final(question):
    decision_item = corrective_rag_decision(question)

    route = decision_item["route"]
    decision = decision_item["decision"]
    reason = decision_item["reason"]
    retrieved_df = decision_item["retrieved_sources"]

    sources = build_source_records(retrieved_df) if not retrieved_df.empty else []
    evidence = build_evidence_records(retrieved_df) if not retrieved_df.empty else []

    if decision == "refuse":
        answer = (
            "Tôi chưa tìm thấy đủ thông tin phù hợp trong phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống. "
            "Vì vậy tôi không nên trả lời để tránh suy đoán sai."
        )

        return {
            "question": question,
            "route": route,
            "decision": decision,
            "reason": reason,
            "should_call_llm": False,
            "final_response_type": "direct_response",
            "answer": answer,
            "num_sources": len(retrieved_df),
            "source_distribution": decision_item["source_distribution"],
            "sources": sources,
            "evidence": evidence,
            "prompt": None
        }

    if decision == "ask_clarification":
        answer = (
            "Bạn vui lòng nói rõ hơn chính sách hoặc vấn đề cụ thể cần hỏi. "
            "Ví dụ: chính sách thiết bị làm việc, lương thử việc, bảo hiểm, hợp đồng lao động, "
            "hoặc quy định áp dụng cho nhân viên Việt Nam."
        )

        return {
            "question": question,
            "route": route,
            "decision": decision,
            "reason": reason,
            "should_call_llm": False,
            "final_response_type": "direct_response",
            "answer": answer,
            "num_sources": len(retrieved_df),
            "source_distribution": decision_item["source_distribution"],
            "sources": sources,
            "evidence": evidence,
            "prompt": None
        }

    answer = (
        "Hệ thống đã tìm thấy context phù hợp và sẽ tạo prompt RAG để LLM sinh câu trả lời có nguồn."
    )

    prompt = build_corrective_prompt_from_sources(
        question=question,
        route=route,
        decision=decision,
        reason=reason,
        retrieved_df=retrieved_df
    )

    return {
        "question": question,
        "route": route,
        "decision": decision,
        "reason": reason,
        "should_call_llm": True,
        "final_response_type": "rag_prompt",
        "answer": answer,
        "num_sources": len(retrieved_df),
        "source_distribution": decision_item["source_distribution"],
        "sources": sources,
        "evidence": evidence,
        "prompt": prompt
    }


final_pipeline_outputs = [corrective_rag_pipeline_final(q) for q in test_questions]

final_pipeline_summary_df = pd.DataFrame([
    {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "final_response_type": item["final_response_type"],
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"]
    }
    for item in final_pipeline_outputs
])

final_pipeline_path = PROMPT_DIR / "corrective_rag_pipeline_final_demo.json"

with open(final_pipeline_path, "w", encoding="utf-8") as f:
    json.dump(final_pipeline_outputs, f, ensure_ascii=False, indent=2)

cross_policy_prompt_final_path = PROMPT_DIR / "latest_corrective_cross_policy_prompt_final.txt"

for item in final_pipeline_outputs:
    if item["route"] == "cross_policy" and item["should_call_llm"]:
        cross_policy_prompt_final_path.write_text(item["prompt"], encoding="utf-8")
        break

print("Đã lưu demo final:", final_pipeline_path)
print("Đã lưu prompt CrossPolicy final:", cross_policy_prompt_final_path)

display(final_pipeline_summary_df)

for item in final_pipeline_outputs:
    print("=" * 100)
    print("QUESTION:", item["question"])
    print("ROUTE:", item["route"])
    print("DECISION:", item["decision"])
    print("SHOULD CALL LLM:", item["should_call_llm"])

    if item["should_call_llm"]:
        print("PROMPT PREVIEW:")
        print(item["prompt"][:1500])
    else:
        print("DIRECT RESPONSE:")
        print(item["answer"])

Đã lưu demo final: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\corrective_rag_pipeline_final_demo.json
Đã lưu prompt CrossPolicy final: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_corrective_cross_policy_prompt_final.txt


,question,route,decision,should_call_llm,final_response_type,num_sources,source_distribution
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,rag_prompt,6,"{'company_handbook': 3, 'legal': 3}"
1,What is the company policy for managing work d...,company_only,answer,True,rag_prompt,5,{'company_handbook': 5}
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,rag_prompt,5,{'legal': 5}
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,direct_response,0,{}
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,direct_response,0,{}


QUESTION: Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
ROUTE: cross_policy
DECISION: answer_with_legal_review_notice
SHOULD CALL LLM: True
PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

ROUTE:
cross_policy

DECISION:
answer_with_legal_review_notice

LÝ DO DECISION:
Có cả nguồn nội bộ và nguồn pháp luật liên quan, nhưng vẫn cần cảnh báo HR/pháp chế kiểm tra trước khi áp dụng chính thức.

NHIỆM VỤ:
Trả lời đối chiếu giữa nguồn nội bộ và nguồn pháp luật. Bắt buộc nhắc HR/pháp chế kiểm tra trước khi áp dụng chính thức.

QUY TẮC BẮT BUỘC:
- Chỉ dùng thông tin trong CONTEXT.
- Không bịa thêm điều khoản, luật, chính sách hoặc con số không có trong CONTEXT.
- Nếu nguồn pháp luật chưa đủ chắc, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Câu trả lời phải dễ hiểu, có cấu trúc và có nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính s

# Cell 08 - Tạo answer preview cuối cùng cho Corrective RAG

## Mục tiêu cell này
Tạo câu trả lời demo cuối cùng cho các case Corrective RAG.

## Vì sao cần làm bước này?
Ở Cell 07, hệ thống đã tạo được prompt cho các câu được phép trả lời.  
Tuy nhiên, để thuyết trình hoặc demo app, ta cần nhìn thấy output cuối cùng ở dạng người dùng dễ hiểu.

Trong app thật:
- Nếu `should_call_llm = True`, LLM sẽ sinh câu trả lời từ prompt.
- Nếu `should_call_llm = False`, hệ thống trả lời trực tiếp bằng rule.

Ở cell này, ta tạo answer preview bằng template có kiểm soát để mô phỏng output cuối.

## Giải thích code
Code sẽ:
1. Nhận output từ `corrective_rag_pipeline_final()`
2. Nếu là `cross_policy`, tạo câu trả lời đối chiếu nội bộ + pháp luật
3. Nếu là `company_only`, tạo câu trả lời dựa trên handbook
4. Nếu là `legal_only`, tạo câu trả lời dựa trên legal evidence
5. Nếu là `refuse` hoặc `ask_clarification`, dùng direct response
6. Lưu kết quả vào JSON

## Output mong đợi
Bạn cần thấy 5 câu hỏi mẫu có câu trả lời preview tương ứng.

In [9]:
def build_corrective_answer_preview(item):
    question = item["question"]
    route = item["route"]
    decision = item["decision"]

    if not item["should_call_llm"]:
        return item["answer"]

    if route == "cross_policy":
        return (
            "Theo nguồn handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, "
            "bao gồm cấu hình bảo mật thiết bị, kiểm soát truy cập hệ thống nội bộ, "
            "không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi bị mất "
            "hoặc khi nhân viên rời công ty. Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm "
            "với quy định/quy chế về bàn giao thiết bị, quản lý tài sản, sử dụng thiết bị đúng mục đích, "
            "bảo mật dữ liệu và trách nhiệm của người lao động khi sử dụng tài sản công ty. "
            "Do đây là vấn đề có yếu tố pháp lý, HR/pháp chế cần kiểm tra lại hợp đồng lao động, "
            "biên bản bàn giao thiết bị, quy chế tài sản nội bộ và quy định lao động hiện hành trước khi áp dụng chính thức."
        )

    if route == "company_only":
        return (
            "Theo handbook nội bộ, chính sách quản lý thiết bị làm việc yêu cầu thiết bị công ty được cấu hình và bảo mật, "
            "ví dụ mã hóa ổ đĩa, firewall, quy tắc mật khẩu và cập nhật bảo mật. "
            "Code, secrets, VPN và hệ thống nội bộ chỉ nên được sử dụng trên thiết bị làm việc đáng tin cậy, "
            "không lưu trên thiết bị cá nhân. Công ty cũng có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty."
        )

    if route == "legal_only" and "lương thử việc" in question.lower():
        return (
            "Theo nguồn pháp luật được truy xuất, tiền lương trong thời gian thử việc do hai bên thỏa thuận, "
            "nhưng ít nhất phải bằng 85% mức lương của công việc đó. "
            "Ngoài ra, các nguồn liên quan cũng nhắc đến quy định về thử việc, thời gian thử việc và hành vi vi phạm quy định về thử việc."
        )

    if route == "legal_only":
        return (
            "Tôi tìm thấy nguồn pháp luật/tài liệu Việt Nam liên quan. "
            "Câu trả lời cần được xây dựng dựa trên các nguồn đã truy xuất và không suy đoán ngoài tài liệu."
        )

    return item["answer"]


final_preview_outputs = []

for item in final_pipeline_outputs:
    preview_item = {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "final_response_type": item["final_response_type"],
        "answer_preview": build_corrective_answer_preview(item),
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"],
        "sources": item["sources"],
        "evidence": item["evidence"]
    }

    final_preview_outputs.append(preview_item)

corrective_preview_path = PROMPT_DIR / "corrective_rag_answer_preview_final.json"

with open(corrective_preview_path, "w", encoding="utf-8") as f:
    json.dump(final_preview_outputs, f, ensure_ascii=False, indent=2)

print("Đã lưu:", corrective_preview_path)

for item in final_preview_outputs:
    print("=" * 100)
    print("QUESTION:", item["question"])
    print("ROUTE:", item["route"])
    print("DECISION:", item["decision"])
    print("SHOULD CALL LLM:", item["should_call_llm"])
    print("ANSWER PREVIEW:")
    print(item["answer_preview"])
    print("NUM SOURCES:", item["num_sources"])
    print("SOURCE DISTRIBUTION:", item["source_distribution"])

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\corrective_rag_answer_preview_final.json
QUESTION: Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
ROUTE: cross_policy
DECISION: answer_with_legal_review_notice
SHOULD CALL LLM: True
ANSWER PREVIEW:
Theo nguồn handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm cấu hình bảo mật thiết bị, kiểm soát truy cập hệ thống nội bộ, không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty. Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với quy định/quy chế về bàn giao thiết bị, quản lý tài sản, sử dụng thiết bị đúng mục đích, bảo mật dữ liệu và trách nhiệm của người lao động khi sử dụng tài sản công ty. Do đây là vấn đề có yếu tố pháp lý, HR/pháp chế cần kiểm tra lại hợp đồng lao động, biên bản bàn giao thiết bị, quy chế tài sản n

# Cell 09 - Lưu summary và kiểm tra cuối notebook Corrective RAG

## Mục tiêu cell này
Lưu lại kết quả quan trọng của notebook `09_corrective_rag.ipynb` và kiểm tra toàn bộ file đầu ra.

## Vì sao cần làm bước này?
Notebook 09 đã xây dựng Corrective RAG, tức là hệ thống không chỉ truy xuất context mà còn biết quyết định:
- khi nào nên trả lời
- khi nào nên trả lời kèm cảnh báo HR/pháp chế
- khi nào nên hỏi lại người dùng
- khi nào nên từ chối vì ngoài phạm vi

Đây là bước rất quan trọng để hệ thống RAG doanh nghiệp an toàn hơn, tránh bịa hoặc trả lời khi context yếu.

## Kết quả chính của notebook này
Corrective RAG xử lý được 5 nhóm câu hỏi:
1. `cross_policy`: đối chiếu chính sách nội bộ với pháp luật Việt Nam
2. `company_only`: chỉ hỏi handbook nội bộ
3. `legal_only`: chỉ hỏi pháp luật/tài liệu Việt Nam
4. `out_of_scope`: câu hỏi ngoài phạm vi
5. `need_clarification`: câu hỏi mơ hồ cần hỏi lại

## Giải thích code
Code sẽ:
1. Lưu summary của Corrective RAG vào file JSON
2. Kiểm tra các file demo/prompt/preview đã tạo
3. Hiển thị bảng trạng thái file OK/MISSING
4. Hiển thị lại bảng demo final

## Output mong đợi
Tất cả file quan trọng của notebook 09 cần có trạng thái `OK`.

In [10]:
corrective_rag_summary = {
    "notebook": "09_corrective_rag.ipynb",
    "main_goal": "Build Corrective RAG decision layer on top of Reranker RAG.",
    "best_retrieval_pipeline": {
        "company_handbook": "Dense Retrieval",
        "legal": "Weighted Hybrid Retrieval + Reranker",
        "reranker_model": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
        "embedding_model": "intfloat/multilingual-e5-base"
    },
    "supported_routes": {
        "cross_policy": "Questions requiring both internal handbook and Vietnamese legal context.",
        "company_only": "Questions answered using company handbook only.",
        "legal_only": "Questions answered using legal corpus only.",
        "out_of_scope": "Questions outside enterprise/legal scope; should be refused.",
        "need_clarification": "Vague questions; system should ask user to clarify."
    },
    "decisions": {
        "answer": "Context is strong enough to answer.",
        "answer_with_caution": "Context exists but is not strong enough for confident conclusion.",
        "answer_with_legal_review_notice": "Answer is allowed but must warn HR/legal review is needed.",
        "ask_clarification": "Question is too vague and needs clarification.",
        "refuse": "Question is out of scope or context is insufficient."
    },
    "demo_cases": [
        {
            "question": item["question"],
            "route": item["route"],
            "decision": item["decision"],
            "should_call_llm": item["should_call_llm"],
            "num_sources": item["num_sources"],
            "source_distribution": item["source_distribution"]
        }
        for item in final_pipeline_outputs
    ],
    "important_outputs": {
        "corrective_demo_outputs": str(PROMPT_DIR / "corrective_rag_demo_outputs.json"),
        "corrective_pipeline_final_demo": str(PROMPT_DIR / "corrective_rag_pipeline_final_demo.json"),
        "corrective_answer_preview_final": str(PROMPT_DIR / "corrective_rag_answer_preview_final.json"),
        "latest_corrective_cross_policy_prompt_final": str(PROMPT_DIR / "latest_corrective_cross_policy_prompt_final.txt")
    }
}

corrective_summary_path = METRIC_DIR / "corrective_rag_summary.json"

with open(corrective_summary_path, "w", encoding="utf-8") as f:
    json.dump(corrective_rag_summary, f, ensure_ascii=False, indent=2)

required_corrective_files = [
    PROMPT_DIR / "corrective_rag_demo_outputs.json",
    PROMPT_DIR / "corrective_rag_pipeline_final_demo.json",
    PROMPT_DIR / "corrective_rag_answer_preview_final.json",
    PROMPT_DIR / "latest_corrective_cross_policy_prompt_final.txt",
    METRIC_DIR / "corrective_rag_summary.json"
]

corrective_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0
    }
    for path in required_corrective_files
])

final_corrective_demo_df = pd.DataFrame([
    {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "final_response_type": item["final_response_type"],
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"]
    }
    for item in final_pipeline_outputs
])

print("Đã lưu summary:", corrective_summary_path)

print("\nCorrective RAG final demo:")
display(final_corrective_demo_df)

print("\nCorrective RAG output files:")
display(corrective_check_df)

print("Tổng file OK:", (corrective_check_df["status"] == "OK").sum(), "/", len(corrective_check_df))

Đã lưu summary: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\corrective_rag_summary.json

Corrective RAG final demo:


,question,route,decision,should_call_llm,final_response_type,num_sources,source_distribution
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,rag_prompt,6,"{'company_handbook': 3, 'legal': 3}"
1,What is the company policy for managing work d...,company_only,answer,True,rag_prompt,5,{'company_handbook': 5}
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,rag_prompt,5,{'legal': 5}
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,direct_response,0,{}
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,direct_response,0,{}



Corrective RAG output files:


,file,status,size_kb
0,artifacts\prompts\corrective_rag_demo_outputs....,OK,41.22
1,artifacts\prompts\corrective_rag_pipeline_fina...,OK,40.87
2,artifacts\prompts\corrective_rag_answer_previe...,OK,19.80
3,artifacts\prompts\latest_corrective_cross_poli...,OK,9.39
4,artifacts\metrics\corrective_rag_summary.json,OK,3.45


Tổng file OK: 5 / 5



## 1. File 09 làm gì?

File 09 xây thêm tầng **Corrective RAG**.

Ở file 08, pipeline mạnh nhất là:

```text
Question
→ Retrieval
→ Reranker
→ lấy context tốt
→ tạo prompt
```

Nhưng file 09 thêm một câu hỏi quan trọng hơn:

```text
Có nên trả lời câu hỏi này không?
```

Vì trong hệ thống doanh nghiệp, không phải câu nào cũng nên trả lời. Có câu hỏi ngoài phạm vi, có câu quá mơ hồ, có câu cần HR/pháp chế kiểm tra thêm.

---

## 2. Corrective RAG là gì?

Corrective RAG là RAG có khả năng **tự kiểm tra và sửa hành vi trước khi trả lời**.

Nó quyết định:

```text
Nếu context đủ tốt → tạo prompt cho LLM trả lời
Nếu context yếu → trả lời thận trọng hoặc từ chối
Nếu câu hỏi mơ hồ → hỏi lại người dùng
Nếu câu hỏi ngoài phạm vi → không gọi LLM
```

Nói dễ hiểu:
File 08 giúp hệ thống **tìm tài liệu tốt hơn**.
File 09 giúp hệ thống **biết khi nào nên trả lời và khi nào không nên trả lời**.

---

## 3. File 09 dùng lại pipeline nào?

File 09 dùng lại chiến lược tốt nhất từ file 08:

```text
company_handbook → Dense Retrieval
legal → Weighted Hybrid Retrieval + Reranker
```

Lý do:

```text
company_handbook là tiếng Anh
người dùng có thể hỏi tiếng Việt
→ Dense multilingual phù hợp hơn

legal là tiếng Việt
→ Weighted Hybrid + Reranker chọn nguồn pháp luật tốt hơn
```

---

## 4. File 09 phân loại câu hỏi như thế nào?

File 09 tạo `Question Router`, tức là bộ phân loại câu hỏi.

Có 5 route chính:

```text
cross_policy
company_only
legal_only
out_of_scope
need_clarification
```

Ý nghĩa:

```text
cross_policy
→ câu hỏi cần đối chiếu chính sách nội bộ + pháp luật Việt Nam

company_only
→ câu hỏi chỉ cần handbook nội bộ

legal_only
→ câu hỏi chỉ cần nguồn pháp luật

out_of_scope
→ câu hỏi ngoài phạm vi hệ thống

need_clarification
→ câu hỏi mơ hồ, cần hỏi lại
```

---

## 5. Ví dụ route trong file 09

Câu hỏi:

```text
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
```

Route:

```text
cross_policy
```

Vì câu này cần cả:

```text
company_handbook: chính sách thiết bị làm việc
legal: quy định/tài liệu Việt Nam liên quan
```

---

Câu hỏi:

```text
What is the company policy for managing work devices?
```

Route:

```text
company_only
```

Vì câu này chỉ hỏi policy nội bộ công ty.

---

Câu hỏi:

```text
Lương thử việc được quy định như thế nào?
```

Route:

```text
legal_only
```

Vì câu này hỏi quy định pháp luật.

---

Câu hỏi:

```text
Cách nấu phở bò ngon tại nhà như thế nào?
```

Route:

```text
out_of_scope
```

Vì câu này không thuộc doanh nghiệp/pháp luật.

---

Câu hỏi:

```text
Chính sách này áp dụng sao?
```

Route:

```text
need_clarification
```

Vì không rõ “chính sách này” là chính sách nào.

---

## 6. File 09 có những decision nào?

Sau khi phân loại route, hệ thống đưa ra decision:

```text
answer
answer_with_caution
answer_with_legal_review_notice
ask_clarification
refuse
```

Ý nghĩa:

```text
answer
→ có đủ context, có thể trả lời

answer_with_caution
→ có context nhưng chưa đủ mạnh, trả lời thận trọng

answer_with_legal_review_notice
→ có thể trả lời, nhưng phải nhắc HR/pháp chế kiểm tra

ask_clarification
→ câu hỏi mơ hồ, hỏi lại người dùng

refuse
→ câu hỏi ngoài phạm vi hoặc không có tài liệu phù hợp
```

---

## 7. Kết quả demo cuối của file 09

Bạn chạy được bảng cuối:

```text
cross_policy       → answer_with_legal_review_notice → gọi LLM
company_only       → answer                         → gọi LLM
legal_only         → answer                         → gọi LLM
out_of_scope       → refuse                         → không gọi LLM
need_clarification → ask_clarification              → không gọi LLM
```

Đây là kết quả rất đúng.

Nghĩa là hệ thống không còn kiểu:

```text
Câu nào cũng đưa vào LLM để trả lời
```

mà đã biết kiểm soát:

```text
Câu nào nên trả lời
Câu nào nên hỏi lại
Câu nào nên từ chối
```

---

## 8. Vì sao `out_of_scope` không gọi LLM?

Ví dụ:

```text
Cách nấu phở bò ngon tại nhà như thế nào?
```

Nếu đưa câu này vào RAG, hệ thống vẫn có thể retrieve nhầm vài tài liệu gần nhất rồi LLM trả lời bịa.

Corrective RAG ngăn điều đó bằng cách:

```text
route = out_of_scope
decision = refuse
should_call_llm = False
```

Output:

```text
Tôi chưa tìm thấy đủ thông tin phù hợp trong phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống. Vì vậy tôi không nên trả lời để tránh suy đoán sai.
```

Đây là hành vi an toàn.

---

## 9. Vì sao `need_clarification` không gọi LLM?

Ví dụ:

```text
Chính sách này áp dụng sao?
```

Câu này thiếu ngữ cảnh. Nếu retrieve ngay thì hệ thống không biết “chính sách này” là gì.

Nên Corrective RAG quyết định:

```text
route = need_clarification
decision = ask_clarification
should_call_llm = False
```

Output:

```text
Bạn vui lòng nói rõ hơn chính sách hoặc vấn đề cụ thể cần hỏi.
```

Đây là đúng hành vi của chatbot doanh nghiệp.

---

## 10. File 09 tạo prompt như thế nào?

Với các câu được phép trả lời, file 09 tạo prompt có thêm thông tin:

```text
ROUTE
DECISION
LÝ DO DECISION
NHIỆM VỤ
QUY TẮC BẮT BUỘC
QUESTION
CONTEXT
YÊU CẦU OUTPUT
```

Ví dụ với `cross_policy`, prompt ghi rõ:

```text
Trả lời đối chiếu giữa nguồn nội bộ và nguồn pháp luật.
Bắt buộc nhắc HR/pháp chế kiểm tra trước khi áp dụng chính thức.
```

Điểm hay là LLM không chỉ nhận context, mà còn biết **mức độ an toàn khi trả lời**.

---

## 11. File 09 đã tối ưu gì?

Ban đầu hàm tạo prompt bị gọi lại retrieval lần nữa.

Tức là:

```text
retrieve lần 1 → decision
retrieve lần 2 → tạo prompt
```

Điều này chậm vì reranker rất tốn thời gian.

Sau đó mình sửa thành:

```text
retrieve một lần
→ decision
→ dùng chính sources đó để tạo prompt
```

Đây là pipeline tối ưu hơn cho app thật.

---

## 12. Answer preview cuối cùng có ý nghĩa gì?

File 09 tạo answer preview để demo trước khi gắn LLM thật.

Ví dụ câu cross-policy:

```text
Theo nguồn handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc...
Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm...
HR/pháp chế cần kiểm tra lại...
```

Câu này cho thấy hệ thống biết:

```text
dùng nguồn nội bộ
dùng nguồn legal
không kết luận pháp lý quá đà
cảnh báo cần kiểm tra trước khi áp dụng
```

Đây là đúng tinh thần CrossPolicyRAG.

---

## 13. File 09 đã tạo các file nào?

Bạn kiểm tra cuối thấy:

```text
corrective_rag_demo_outputs.json              OK
corrective_rag_pipeline_final_demo.json       OK
corrective_rag_answer_preview_final.json      OK
latest_corrective_cross_policy_prompt_final.txt OK
corrective_rag_summary.json                   OK
```

Tổng:

```text
5 / 5 file OK
```

Nghĩa là file 09 đã hoàn tất.

---

## 14. Chốt file 09 ngắn gọn

File 09 biến RAG từ:

```text
tìm context rồi trả lời
```

thành:

```text
hiểu loại câu hỏi
→ truy xuất nguồn phù hợp
→ kiểm tra context
→ quyết định có nên gọi LLM không
→ tạo prompt hoặc trả lời trực tiếp
```

Đây là bước rất quan trọng để hệ thống RAG doanh nghiệp an toàn hơn.

## 15. Vì sao file 10 là bước tiếp theo?

Sau file 09, mình đã có pipeline khá đầy đủ:

```text
Dense / Hybrid / Reranker / Corrective Gate
```

File 10 hợp lý để làm:

```text
Generation / LLM Answering
```

Tức là bắt đầu gắn LLM thật hoặc mô phỏng generation chính thức:

```text
Corrective RAG prompt
→ LLM sinh câu trả lời
→ kiểm tra answer có nguồn
→ lưu kết quả demo/evaluation
```

Nói ngắn gọn:

```text
File 09 quyết định có nên trả lời không.
File 10 sẽ sinh câu trả lời cuối cùng.
```
